# Linear Regression — Taxi Fare Prediction

**Name:** K Jnanendra  
**Roll No:** CH.SC.U4CSE24217

This notebook implements the Linear Regression taxi-fare exercise from the ML Lab Activity Book.

**Dataset requirement:** Save the Chicago Taxi Trips dataset as `chicago_taxi_trips.csv` in the same folder. If your dataset uses different column names, edit the candidate lists below.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
CSV_FILE = "chicago_taxi_trips.csv"
FARE_CANDIDATES = ["fare", "trip_total", "trip fare", "fare_amount"]
FEATURE_CANDIDATES = [
    "trip_miles", "trip_seconds", "pickup_community_area",
    "dropoff_community_area", "payment_type"
]

df = pd.read_csv(CSV_FILE)
df.columns = [c.strip() for c in df.columns]
lower_to_actual = {c.lower(): c for c in df.columns}
fare_col = next((lower_to_actual[c] for c in FARE_CANDIDATES if c in lower_to_actual), None)
if fare_col is None:
    raise ValueError(f"Could not identify fare target. Available columns: {df.columns.tolist()}. Update FARE_CANDIDATES.")
feature_cols = [lower_to_actual[c] for c in FEATURE_CANDIDATES if c in lower_to_actual and lower_to_actual[c] != fare_col]
if not feature_cols:
    raise ValueError(f"Could not identify predictors. Available columns: {df.columns.tolist()}. Update FEATURE_CANDIDATES.")

data = df[feature_cols + [fare_col]].copy()
data[fare_col] = pd.to_numeric(data[fare_col], errors="coerce")
data = data.dropna(subset=[fare_col])
data = data[data[fare_col] >= 0]
X = data[feature_cols]
y = data[fare_col]
numeric_cols = X.select_dtypes(include="number").columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
categorical_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([("num", numeric_pipe, numeric_cols), ("cat", categorical_pipe, categorical_cols)])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = Pipeline([("preprocessor", preprocessor), ("regressor", LinearRegression())])
model.fit(X_train, y_train)
pred = model.predict(X_test)

print("Target column:", fare_col)
print("Features:", feature_cols)
print("Training rows:", len(X_train), "| Testing rows:", len(X_test))
print("MAE:", mean_absolute_error(y_test, pred))
print("MSE:", mean_squared_error(y_test, pred))
print("RMSE:", mean_squared_error(y_test, pred) ** 0.5)
print("R2 score:", r2_score(y_test, pred))

In [ ]:
plt.scatter(y_test, pred, alpha=0.5)
plt.xlabel("Actual fare")
plt.ylabel("Predicted fare")
plt.title("Actual vs Predicted Taxi Fare")
plt.tight_layout()
plt.show()